In [ ]:
from openai import OpenAI
import os
token = '' # Add your token here 
client = OpenAI(api_key=token)

import pymupdf
import os


def summarization_gen(instruction, model_temp, model_name):
    response = client.chat.completions.create(
        model=model_name,  # Or the specific model you have access to
        messages=[
            {"role": "system", "content": "You are a scientific assistant specializing in molecular biology and bioinformatics. Your goal is to generate concise, structured gene descriptions based on extracted scientific data."},
            {"role": "user", "content": f"{instruction}"}
        ],
    )
    
    # Extract the response
    # print(response)
    information = response.choices[0].message.content.strip()
    
    return information



def create_prompt_from_text(existing_description, extracted_info_json):
    instruction = f"""
## **Task Description**
Update the existing gene description using the extracted information below. Maintain the structure and clarity of the original description while integrating newly extracted information. Expand on incomplete sections and refine content as needed.

The existing description and extracted facts may contain specific biological relationship phrases and evidence qualifiers. Preserve these relationships and evidence status when updating the description. Do not reinterpret a fact as predicted or factual during summary generation.

## **Existing Gene Description**:
{existing_description}

## **Extracted Information**:
{extracted_info_json}

## **Guidelines for Updating the Description**
- **Preserve and refine**: Keep relevant information from the existing description and improve readability where necessary.
- **Incorporate newly extracted information**: Use the extracted information to enhance the summary, ensuring all relevant sections are updated.
- **Preserve existing evidence**: Do not downgrade an existing unqualified statement to a predicted statement only because a newly extracted fact about the same relationship is predicted.
- **Summarize effectively**: Group multiple terms into **high-level categories** to avoid excessive listing.
- **Follow structured formatting**: Adhere to the sentence templates below while preserving the relationship phrases and evidence status from the existing description and extracted facts.

## **Sentence Templates for Gene Description**:
- **Molecular Function**: Preserve the relationship phrase from the existing description or extracted facts, such as:
  - Enables <list of gene ontology molecular function terms>.
  - Predicted to enable <list of gene ontology molecular function terms>.
  - Contributes to <list of gene ontology molecular function terms>.
  - Predicted to contribute to <list of gene ontology molecular function terms>.

- **Biological Process**: Preserve the relationship phrase from the existing description or extracted facts, such as:
  - Involved in <list of biological processes>.
  - Predicted to be involved in <list of biological processes>.
  - Acts upstream of or within <list of biological processes>.
  - Predicted to act upstream of or within <list of biological processes>.

- **Subcellular Localization**: Preserve the relationship phrase from the existing description or extracted facts, such as:
  - Is active in <list of subcellular locations>.
  - Predicted to be active in <list of subcellular locations>.
  - Is located in <list of subcellular locations>.
  - Predicted to be located in <list of subcellular locations>.
  - Is part of <list of protein-containing complexes>.
  - Predicted to be part of <list of protein-containing complexes>.

- **Expression**: Preserve the relationship phrase from the existing description or extracted facts:
  - Is expressed in <list of anatomy ontology terms>.
  - Predicted to be expressed in <list of anatomy ontology terms>.

- **Pathways**: Preserve the relationship phrase from the extracted facts:
  - Is part of <list of pathway terms>.
  - Predicted to be part of <list of pathway terms>.

- **Interaction**: Preserve the relationship phrase from the extracted facts:
  - Interacts with <list of genes/proteins>.
  - Predicted to interact with <list of genes/proteins>.

- **Disease Relevance**:
  - Used to study <list of disease ontology terms>.

- **Implication of Human Orthologs**:
  - Human ortholog(s) of this gene implicated in <list of disease ontology terms>.

- **Human Orthologs**:
  - Orthologous to <list of human orthologs>.

- **Modifiers**:
  - <Modifier> ameliorates or exacerbates the disease phenotype or condition.

## **Rules for Updating and Summarization**
- Maintain a **scientific tone** and avoid redundant phrases.
- Ensure **grammatical correctness**.
- Use **semicolons (;)** for separating multiple terms when appropriate.
- **Omit** fields with no available data.

- **Preserve the relationship phrase and evidence status of each fact.**
- If an extracted fact contains **"predicted"**, any newly added statement based on that fact must retain **"predicted"**.
- Never convert a predicted statement into an unqualified factual statement.
- Never convert an existing unqualified factual statement into a predicted statement solely because a newly extracted fact is predicted.
- Preserve distinctions between biological relationships:
  - `enables` vs. `contributes to`
  - `is involved in` vs. `acts upstream of or within`
  - `is active in` vs. `is located in` vs. `is part of`
  - and their corresponding predicted forms.
- When multiple facts have the **same relationship phrase and evidence status**, they may be combined into one sentence.
- When facts have **different relationship phrases or different evidence status**, keep them separate.
- If the existing description and newly extracted information describe the **same fact**, avoid duplication. Preserve the existing statement unless the extracted information provides additional compatible information that should be added.
- Summarization may group or shorten biological terms, but must not change their associated relationship or evidence status.

Using the existing gene description and structured extracted information, generate an updated, complete, and well-formatted gene description.
"""

# THe template for Molecular Function needs to be changed (Enables)
# Group multiple child terms into higher-level parent terms using the relevant GO ontology
    return instruction

In [ ]:
import os
import json
from openai import OpenAI
from pprint import pprint

with open('data/data_pairs_20251002.json', 'r') as f:
    gene_data = json.load(f)

folder = 'gene_description/incomplete_desc_2026'
models = ['o1'] #['gpt-4-turbo', 'o1', 'gpt-5', 'gpt-4o']

for gene_name in gene_data.keys():
    task_name = gene_name+ '_InCompleteDesc_paperinfo_20260927.json'
    if os.path.join(folder, task_name):
        print(gene_name)
        paper_info = {}
        paper_info['gene_name'] = gene_name
        for key in gene_data[gene_name]['paper_inf'].keys():
            if len(gene_data[gene_name]['paper_inf'][key])>0:
                print(gene_data[gene_name]['paper_inf'][key])
                for item in gene_data[gene_name]['paper_inf'][key]:
                    paper_info[key] = [item['sentence template'] for item in gene_data[gene_name]['paper_inf'][key] if 'sentence template' in item.keys()]
        instruction = create_prompt_from_text(gene_data[gene_name]['gt']['description'], paper_info)
        # print(instruction) 

        model_temp = 0.0
        
        gene_info = {}
        for model_name in models:
            gene_info[model_name] = summarization_gen(instruction, model_temp, model_name)
        # print('*'*10)
        # print(gene_info)
        # print('*'*10)
        task_name = gene_name+ '_InCompleteDesc_paperinfo_20260927.json'
        with open(os.path.join(folder, task_name), 'w') as f:
            json.dump(gene_info, f, indent=4)